# LAB 10 — Extractive Question Answering với SQuAD

**Mục tiêu:** fine-tune pretrained Transformer để dự đoán start/end của answer trong context; so sánh độ dài 128/256/384, baseline, EM và F1, answer length, lexical overlap và lỗi boundary. Các chỉ số chỉ được xuất sau khi chạy thực tế. Hãy tự viết prediction và calculations **trước** khi chạy notebook.

**Đầu ra:** `results.csv`, `learning_curves.png`, `em_f1_analysis.png`, `answer_length_analysis.png`, `error_analysis.md`.

## 0. Cài đặt và import

Chạy notebook trong thư mục `lab10/` với `qa_model.py` ở cùng cấp. Colab GPU được khuyến nghị. Phiên bản thư viện được giới hạn ở Transformers 4.x để ổn định API.

In [ ]:
%pip -q install 'transformers>=4.47,<5' 'datasets>=3,<5' 'accelerate>=0.26' pandas matplotlib numpy

In [ ]:
import os, re, time, json, random, platform, tempfile
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import transformers, datasets
from datasets import load_dataset
from transformers import (AutoTokenizer, AutoModelForQuestionAnswering,
                          Trainer, TrainingArguments, DefaultDataCollator, set_seed)
from qa_model import (prepare_train_features, prepare_eval_features,
                      postprocess_predictions, keyword_baseline, score_predictions,
                      normalize_answer)

print("Torch:", torch.__version__, "CUDA:", torch.cuda.is_available())
print("Transformers:", transformers.__version__, "Datasets:", datasets.__version__)

## 1. Cấu hình và dữ liệu

Tập train lấy từ SQuAD train. SQuAD 1.1 không cung cấp nhãn test công khai, nên chia validation gốc thành validation/test theo seed cố định. Giữ nguyên các sample khi đổi max length. Mặc định dùng tập con để chạy thực hành; tăng giới hạn nếu có GPU và thời gian.

In [ ]:
SEED = 42
MODEL_NAME = ("distilbert/distilbert-base-uncased" if torch.cuda.is_available()
              else "google/bert_uncased_L-2_H-128_A-2")
LENGTHS = [128, 256, 384]
STRIDE = 64  # nhỏ hơn mọi max_length, giữ overlap khi context dài
TRAIN_LIMIT, VALID_LIMIT, TEST_LIMIT = ((600, 120, 200) if torch.cuda.is_available()
                                         else (120, 40, 100))
BATCH_SIZE, LEARNING_RATE, EPOCHS = 8, 2e-5, 2
MAX_ANSWER_LENGTH = 30
OUT = Path.cwd()
assert (OUT / "qa_model.py").exists(), "Hãy chạy notebook trong thư mục lab10/"
set_seed(SEED)
raw = load_dataset("rajpurkar/squad")
train_examples = raw["train"].shuffle(seed=SEED).select(range(min(TRAIN_LIMIT, len(raw["train"]))))
held_out = raw["validation"].train_test_split(test_size=0.5, seed=SEED)
valid_examples = held_out["train"].select(range(min(VALID_LIMIT, len(held_out["train"]))))
test_examples = held_out["test"].select(range(min(TEST_LIMIT, len(held_out["test"]))))
print("Train / valid / test:", len(train_examples), len(valid_examples), len(test_examples))
print("Example fields:", train_examples.column_names)
print("First answer:", train_examples[0]["answers"])
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
assert tokenizer.is_fast, 'QA cần fast tokenizer để lấy offset_mapping'

## 2. Tokenization và answer alignment

Input có question + context. Offset từ character span được chuyển sang token span; window không chứa answer dùng CLS target. Eval giữ `example_id` và context offsets để ghép dự đoán từ các window. Xem notebook `tokenizer_alignment.ipynb` để tự kiểm tra chi tiết.

In [ ]:
def make_features(max_length):
    train_features = train_examples.map(
        lambda batch: prepare_train_features(batch, tokenizer, max_length, STRIDE),
        batched=True, remove_columns=train_examples.column_names)
    valid_features = valid_examples.map(
        lambda batch: prepare_train_features(batch, tokenizer, max_length, STRIDE),
        batched=True, remove_columns=valid_examples.column_names)
    test_features = test_examples.map(
        lambda batch: prepare_eval_features(batch, tokenizer, max_length, STRIDE),
        batched=True, remove_columns=test_examples.column_names)
    assert len(train_features) >= len(train_examples)
    assert len(test_features) >= len(test_examples)
    print("max_length", max_length, "train windows", len(train_features), "test windows", len(test_features))
    print("input shape:", np.asarray(train_features["input_ids"]).shape)
    return train_features, valid_features, test_features

## 3. Baseline keyword overlap

Baseline chọn câu có nhiều từ khóa chung với question rồi lấy một span ngắn. Đây là heuristic yếu nhưng trả về đúng một đoạn của context để so sánh với model.

In [ ]:
baseline_predictions = {row["id"]: keyword_baseline(row["question"], row["context"])
                        for row in test_examples}
baseline_scores = pd.DataFrame(score_predictions(test_examples, baseline_predictions))
print("Baseline EM/F1:", baseline_scores[["exact_match", "f1"]].mean().to_dict())

## 4. Fine-tune và thí nghiệm context length

Ba lần chạy dùng cùng dữ liệu, seed, model gốc và training arguments; chỉ thay `max_length`. Trainer ghi train/validation loss theo epoch. Đánh giá EM/F1 trên test giữ riêng. Checkpoint huấn luyện được lưu tạm ngoài `lab10/` rồi xóa khi hoàn tất.

In [ ]:
experiment_rows = []
all_histories = {}
all_test_predictions = {}
all_test_spans = {}
data_collator = DefaultDataCollator()
temp_checkpoints = tempfile.TemporaryDirectory(prefix="lab10_checkpoints_")

for max_length in LENGTHS:
    set_seed(SEED)
    train_features, valid_features, test_features = make_features(max_length)
    model = AutoModelForQuestionAnswering.from_pretrained(MODEL_NAME)
    args = TrainingArguments(
        output_dir=str(Path(temp_checkpoints.name) / f"checkpoints_{max_length}"),
        eval_strategy="epoch", save_strategy="epoch", logging_strategy="epoch",
        learning_rate=LEARNING_RATE, per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE, num_train_epochs=EPOCHS,
        weight_decay=0.01, seed=SEED, data_seed=SEED,
        load_best_model_at_end=True, metric_for_best_model="eval_loss",
        greater_is_better=False, save_total_limit=1, report_to="none",
        disable_tqdm=False,
    )
    trainer = Trainer(model=model, args=args, train_dataset=train_features,
                      eval_dataset=valid_features, data_collator=data_collator,
                      processing_class=tokenizer)
    tic = time.perf_counter()
    train_output = trainer.train()
    training_time = time.perf_counter() - tic
    valid_loss = trainer.evaluate()["eval_loss"]
    model_inputs = test_features.remove_columns(["example_id", "offset_mapping"])
    raw_logits = trainer.predict(model_inputs).predictions
    predictions, spans = postprocess_predictions(
        test_examples, test_features, raw_logits,
        max_answer_length=MAX_ANSWER_LENGTH)
    sample_scores = pd.DataFrame(score_predictions(test_examples, predictions))
    row = {"model": "fine_tuned_transformer", "max_length": max_length,
           "train_loss": train_output.training_loss, "validation_loss": valid_loss,
           "em": 100 * sample_scores["exact_match"].mean(),
           "f1": 100 * sample_scores["f1"].mean(),
           "training_time_sec": round(training_time, 2),
           "train_samples": len(train_examples), "validation_samples": len(valid_examples),
           "test_samples": len(test_examples), "seed": SEED}
    experiment_rows.append(row)
    all_histories[max_length] = trainer.state.log_history
    all_test_predictions[max_length] = predictions
    all_test_spans[max_length] = spans
    print(row)
    del trainer, model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
temp_checkpoints.cleanup()

## 5. Results và learning curves

Lưu các giá trị thật từ ba lần huấn luyện; baseline không có training loss/time. Learning curves dùng log theo epoch, không tự tạo điểm dữ liệu dự kiến.

In [ ]:
baseline_row = {"model": "keyword_baseline", "max_length": None,
                "train_loss": None, "validation_loss": None,
                "em": 100 * baseline_scores["exact_match"].mean(),
                "f1": 100 * baseline_scores["f1"].mean(),
                "training_time_sec": 0,
                "train_samples": len(train_examples), "validation_samples": len(valid_examples),
                "test_samples": len(test_examples), "seed": SEED}
results = pd.DataFrame([baseline_row] + experiment_rows)
display(results)
results.to_csv(OUT / "results.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for length, history in all_histories.items():
    train_logs = [h for h in history if "loss" in h and "eval_loss" not in h and "epoch" in h]
    valid_logs = [h for h in history if "eval_loss" in h and "epoch" in h]
    axes[0].plot([h["epoch"] for h in train_logs], [h["loss"] for h in train_logs], marker="o", label=str(length))
    axes[1].plot([h["epoch"] for h in valid_logs], [h["eval_loss"] for h in valid_logs], marker="o", label=str(length))
for axis, title in zip(axes, ["Training loss", "Validation loss"]):
    axis.set(title=title, xlabel="Epoch", ylabel="Loss")
    axis.legend(title="Max length")
    axis.grid(alpha=.3)
fig.tight_layout()
fig.savefig(OUT / "learning_curves.png", dpi=160)
plt.show()

## 6. Exact Match và F1

So sánh baseline với model fine-tuned 384. Liệt kê ít nhất 5 trường hợp `EM=0, F1>0` nếu tập test có đủ. Sinh viên tự giải thích vì sao từng prediction chỉ khớp một phần.

In [ ]:
fine_predictions = all_test_predictions[384]
fine_spans = all_test_spans[384]
fine_scores = pd.DataFrame(score_predictions(test_examples, fine_predictions))
comparison = pd.DataFrame({
    "model": ["Keyword baseline", "Fine-tuned Transformer (384)"],
    "EM": [100 * baseline_scores["exact_match"].mean(), 100 * fine_scores["exact_match"].mean()],
    "F1": [100 * baseline_scores["f1"].mean(), 100 * fine_scores["f1"].mean()]})
display(comparison)
ax = comparison.set_index("model")[["EM", "F1"]].plot.bar(rot=0, figsize=(9, 4))
ax.set(ylabel="Score (%)", title="Exact Match vs token F1")
ax.grid(axis="y", alpha=.3)
plt.tight_layout()
plt.savefig(OUT / "em_f1_analysis.png", dpi=160)
plt.show()

partial = fine_scores.query("exact_match == 0 and f1 > 0").head(5)
print("Partial matches found:", len(partial))
for sample_id in partial["id"]:
    sample = next(row for row in test_examples if row["id"] == sample_id)
    print("Question:", sample["question"], "\nGold:", sample["answers"]["text"],
          "\nPrediction:", fine_predictions[sample_id], "\n")

## 7. Answer length và question/context matching

Nhóm answer theo số token như đề: 1, 2–3, 4–6, 7+. Lexical overlap dùng Jaccard giữa question/context, chia high/low theo median của tập test; báo cáo số mẫu để tránh kết luận từ nhóm quá nhỏ.

In [ ]:
score_by_id = fine_scores.set_index("id")
def answer_length_group(text):
    length = len(re.findall(r"\b\w+\b", text))
    if length <= 1: return "1"
    if length <= 3: return "2-3"
    if length <= 6: return "4-6"
    return "7+"

length_rows = []
lexical_rows = []
for row in test_examples:
    sample_id = row["id"]
    gold = row["answers"]["text"][0]
    length_rows.append({"group": answer_length_group(gold),
                        "em": score_by_id.loc[sample_id, "exact_match"],
                        "f1": score_by_id.loc[sample_id, "f1"]})
    q_words = set(normalize_answer(row["question"]).split())
    c_words = set(normalize_answer(row["context"]).split())
    overlap = len(q_words & c_words) / max(1, len(q_words | c_words))
    lexical_rows.append({"id": sample_id, "overlap": overlap,
                         "answer_word_overlap": len(q_words & set(normalize_answer(gold).split())) > 0,
                         "em": score_by_id.loc[sample_id, "exact_match"],
                         "f1": score_by_id.loc[sample_id, "f1"]})
length_df = pd.DataFrame(length_rows)
length_summary = length_df.groupby("group").agg(n=("f1", "size"), em=("em", "mean"), f1=("f1", "mean"))
length_summary = length_summary.reindex(["1", "2-3", "4-6", "7+"]).dropna(subset=["n"])
display(length_summary)
ax = (100 * length_summary[["em", "f1"]]).plot.bar(rot=0, figsize=(8, 4))
ax.set(xlabel="Answer length (normalized tokens)", ylabel="Score (%)", title="QA performance by answer length")
ax.grid(axis="y", alpha=.3)
plt.tight_layout()
plt.savefig(OUT / "answer_length_analysis.png", dpi=160)
plt.show()
lexical_df = pd.DataFrame(lexical_rows)
median_overlap = lexical_df["overlap"].median()
lexical_df["context_overlap_group"] = np.where(lexical_df["overlap"] >= median_overlap, "high", "low")
display(lexical_df.groupby(["context_overlap_group", "answer_word_overlap"]).agg(
    n=("f1", "size"), em=("em", "mean"), f1=("f1", "mean")))

## 8. Error analysis

Đếm wrong start/end bằng character boundaries của span dự đoán và gold đầu tiên. Bảng lấy tối đa 5 ví dụ mỗi nhóm nếu có. Cột explanation để sinh viên tự điền sau khi đọc context và phân loại nguyên nhân; file chỉ ghi quan sát tự động, không gán nguyên nhân suy đoán.

In [ ]:
error_rows = []
for row in test_examples:
    sid = row["id"]
    pred_start, pred_end = fine_spans[sid]
    if pred_start is None: continue
    gold_candidates = [(start, start + len(text), text)
                       for start, text in zip(row["answers"]["answer_start"], row["answers"]["text"])]
    gold_start, gold_end, gold_text = min(
        gold_candidates,
        key=lambda candidate: ((pred_start != candidate[0]) + (pred_end != candidate[1]),
                               abs(pred_start - candidate[0]) + abs(pred_end - candidate[1])))
    wrong_start = pred_start != gold_start
    wrong_end = pred_end != gold_end
    if not (wrong_start or wrong_end): continue
    if wrong_start and wrong_end: error_type = "wrong span"
    elif wrong_start: error_type = "wrong start"
    else: error_type = "wrong end"
    error_rows.append({"question": row["question"], "gold": gold_text,
                       "prediction": fine_predictions[sid], "error_type": error_type,
                       "gold_char_span": (gold_start, gold_end),
                       "pred_char_span": (pred_start, pred_end), "id": sid})
error_df = pd.DataFrame(error_rows)
counts = error_df["error_type"].value_counts()
print("Wrong start:", sum(x["pred_char_span"][0] != x["gold_char_span"][0] for x in error_rows))
print("Wrong end:", sum(x["pred_char_span"][1] != x["gold_char_span"][1] for x in error_rows))
print("Exclusive categories:\n", counts)
review_groups = [
    ("wrong start", error_df[error_df.apply(
        lambda r: r["pred_char_span"][0] != r["gold_char_span"][0], axis=1)].head(5)),
    ("wrong end", error_df[error_df.apply(
        lambda r: r["pred_char_span"][1] != r["gold_char_span"][1], axis=1)].head(5)),
    ("wrong span", error_df[error_df["error_type"] == "wrong span"].head(5)),
]
selected_errors = pd.concat([part.assign(review_group=group) for group, part in review_groups], ignore_index=True)
display(selected_errors)

lines = ["# LAB 10 — Error analysis", "",
         "Các ví dụ dưới đây được tạo từ dự đoán thật. Sinh viên kiểm tra context và tự điền Explanation.", "",
         f"- Wrong start (kể cả wrong span): {sum(x['pred_char_span'][0] != x['gold_char_span'][0] for x in error_rows)}",
         f"- Wrong end (kể cả wrong span): {sum(x['pred_char_span'][1] != x['gold_char_span'][1] for x in error_rows)}", "",
         "| Review group | Question | Gold | Prediction | Error type | Explanation |", "|---|---|---|---|---|---|"]
for _, r in selected_errors.iterrows():
    values = [r["review_group"], r["question"], r["gold"], r["prediction"], r["error_type"], "Sinh viên điền"]
    lines.append("| " + " | ".join(str(v).replace("|", "\\|").replace("\n", " ") for v in values) + " |")
lines += ["", "## Nhận xét của sinh viên", "", "Giải thích nguyên nhân wrong start và wrong end sau khi xem context gốc.", ""]
(OUT / "error_analysis.md").write_text("\n".join(lines), encoding="utf-8")

## 9. Document QA và reproducibility

`document_qa.py` nhận một context/file có sẵn và tự tải model QA pretrained; không retrieval. Cell này kiểm tra các file xuất và in cấu hình thực tế để tái lập.

In [ ]:
reproducibility = {
    "model": MODEL_NAME, "tokenizer": tokenizer.name_or_path, "dataset": "rajpurkar/squad (SQuAD 1.1)",
    "train_split": f"train subset n={len(train_examples)} seed={SEED}",
    "validation_split": f"validation half n={len(valid_examples)} seed={SEED}",
    "test_split": f"validation other half n={len(test_examples)} seed={SEED}",
    "max_sequence_lengths": LENGTHS, "stride": STRIDE, "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE, "epochs": EPOCHS, "optimizer": "AdamW (Trainer default)",
    "random_seed": SEED, "hardware": torch.cuda.get_device_name(0) if torch.cuda.is_available() else platform.processor(),
    "training_time_sec_by_length": {str(r["max_length"]): r["training_time_sec"] for r in experiment_rows},
    "transformers_version": transformers.__version__, "datasets_version": datasets.__version__}
print(json.dumps(reproducibility, indent=2, ensure_ascii=False))
required_outputs = ["results.csv", "learning_curves.png", "em_f1_analysis.png",
                    "answer_length_analysis.png", "error_analysis.md"]
for name in required_outputs:
    path = OUT / name
    assert path.is_file() and path.stat().st_size > 0, name
    print(path)